In [16]:
from pathlib import Path
from tempfile import TemporaryDirectory

import duckdb
import pandas as pd

raw_dir = Path("dataset/raw")
if not raw_dir.is_dir():
    raw_dir = Path("../dataset/raw")


def write_parquet_atomic(relation, output_path, **parquet_options):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=output_path.parent) as temp_dir:
        temp_path = Path(temp_dir) / output_path.name
        relation.write_parquet(str(temp_path), **parquet_options)
        temp_path.replace(output_path)

In [17]:
feature_definitions = pd.read_csv(raw_dir / "feature_definitions_enhanced.csv")
feature_definitions

,Variable,Description,TrainingTables
0,actualdpd_943P,Days Past Due (DPD) of previous contract (actu...,train_applprev_1
1,actualdpdtolerance_344P,DPD of client with tolerance.,train_static_0
2,addres_district_368M,District of the person's address.,train_person_2
3,addres_role_871L,Role of person's address.,train_person_2
4,addres_zip_823M,Zip code of the address.,train_person_2
...,...,...,...
460,totinstallast1m_4525188A,Total amount of monthly instalments paid in th...,train_static_0
461,twobodfilling_608L,Type of application process.,train_static_0
462,type_25L,Contact type of a person.,train_person_1
463,typesuite_864L,Persons accompanying the client during the loa...,train_static_0


## Join depth-0 training features

The two `train_static_0` files are row shards of one table. Read them together, then left-join both depth-0 tables to `train_base` on `case_id`. Check that each input has unique, non-null keys so the joins keep one row per training case. Cases without a match have nulls for that table's features.

The enriched table is written to `dataset/processed/train_base_depth0.parquet`.

In [18]:
train_dir = raw_dir / "train"
static_paths = sorted(train_dir.glob("train_static_0_*.parquet"))
if not static_paths:
    raise FileNotFoundError(f"No train_static_0 shards found in {train_dir}")

depth0_path = raw_dir.parent / "processed" / "train_base_depth0.parquet"

# Limit parallelism and allow DuckDB to write the joined rows in any order.
with duckdb.connect(config={"threads": 2, "preserve_insertion_order": False}) as con:
    base = con.read_parquet(str(train_dir / "train_base.parquet"))
    static = con.read_parquet([str(path) for path in static_paths])
    static_cb = con.read_parquet(str(train_dir / "train_static_cb_0.parquet"))

    for name, table in (
        ("train_base", base),
        ("train_static_0", static),
        ("train_static_cb_0", static_cb),
    ):
        [(row_count, unique_keys)] = table.aggregate("count(*), count(DISTINCT case_id)").fetchall()
        if row_count != unique_keys:
            raise ValueError(f"{name} must have one row per non-null case_id")

    train_depth0 = base.join(static, "case_id", how="left")
    train_depth0 = train_depth0.join(static_cb, "case_id", how="left")
    write_parquet_atomic(train_depth0, depth0_path)

print(f"Saved {depth0_path.resolve()}")

Saved /home/georgiusbenedikt09/MLOps_Credit_Risk/dataset/processed/train_base_depth0.parquet


## Join depth-1 training features without depth-2 detail tables

Summarize `debitcard`, `deposit`, `other`, and the three tax registry tables separately by `case_id`. Each summary has at most one row per case, so the left joins preserve the grain of the depth-0 output.

- Count the available records from each source. Use zero for a case with no matching rows and leave its other summary fields null.
- Use the mean and maximum of amount fields (i.e. fields name ending with `A`). For `other`, use the mean alone since the current training file has at most one record per case.
- Keep the earliest and latest recorded dates (i.e. fields name ending with `D`)
- Use Count distinct for feature names ending with `M` (i.e. Masking categories).

The enriched table is written to `dataset/processed/train_base_depth1.parquet`.

In [19]:
depth1_sources = ("debitcard", "deposit", "other", "tax_registry_a", "tax_registry_b", "tax_registry_c")
depth1_path = depth0_path.with_name("train_base_depth1.parquet")

with duckdb.connect(config={"threads": 2, "preserve_insertion_order": False}) as con:
    train_depth1 = con.read_parquet(str(depth0_path))
    count_columns = []

    for source in depth1_sources:
        table = con.read_parquet(str(train_dir / f"train_{source}_1.parquet"))
        count_column = f"{source}_record_count"
        count_columns.append(count_column)
        aggregations = ["case_id", f"count(*) AS {count_column}"]
        amount_stats = ("avg",) if source == "other" else ("avg", "max")

        for column in table.columns:
            feature = f"{source}_{column}"
            if column.endswith("A"):
                aggregations.extend(f"{stat}({column}) AS {feature}_{stat}" for stat in amount_stats)
            elif column.endswith("D"):
                aggregations.extend(
                    f"{stat}(CAST({column} AS DATE)) AS {feature}_{stat}" for stat in ("min", "max")
                )
            elif column.endswith("M"):
                aggregations.append(f"count(DISTINCT {column}) AS {feature}_distinct_count")

        features = table.aggregate(", ".join(aggregations), "case_id")
        train_depth1 = train_depth1.join(features, "case_id", how="left")

    zero_missing_counts = ", ".join(f"coalesce({column}, 0) AS {column}" for column in count_columns)
    train_depth1 = train_depth1.project(f"* REPLACE ({zero_missing_counts})")
    write_parquet_atomic(train_depth1, depth1_path)

print(f"Saved {depth1_path.resolve()}")

Saved /home/georgiusbenedikt09/MLOps_Credit_Risk/dataset/processed/train_base_depth1.parquet


## Join depth-1 training features with depth-2 detail tables

Process the previous application, credit bureau A, credit bureau B, and person table pairs. For each pair, group the detail rows by `(case_id, num_group1)`, match those summaries to the depth-1 keys, and then roll up to `case_id`. Summarize depth-1 rows separately so a parent with more details is counted only once.

- Summarize numeric fields with means and maxima, dates and numeric calendar fields with extrema, and categorical fields with distinct counts. For depth-2 numeric means, each parent with a value gets equal weight. For depth-2 categories, keep the largest distinct count within any one parent.
- For `person_1`, retain the applicant's own values (`num_group1 = 0`) and count all people on the case. The depth-2 summaries cover detail records for all matched people.
- Fill missing record counts with zero after the left joins. Leave unavailable statistics null.

The final table, including the features from both earlier steps, is written to `dataset/processed/train_base_depth2.parquet`.

In [ ]:
def read_training_history(con, source, depth):
    stem = f"train_{source}_{depth}"
    single_path = train_dir / f"{stem}.parquet"
    paths = [single_path] if single_path.is_file() else sorted(train_dir.glob(f"{stem}_*.parquet"))
    if not paths:
        raise FileNotFoundError(f"No Parquet files found for {stem} in {train_dir}")
    return con.read_parquet([str(path) for path in paths])


def history_statistics(table, prefix):
    for column, dtype in zip(table.columns, table.types):
        if column in {"case_id", "num_group1", "num_group2"}:
            continue

        if column.endswith("D"):
            value, stats = f"CAST({column} AS DATE)", ("min", "max")
        elif str(dtype) == "VARCHAR":
            yield f"{prefix}_{column}_distinct_count", f"count(DISTINCT {column})", "max"
            continue
        elif str(dtype) == "BOOLEAN":
            yield f"{prefix}_{column}_true_share", f"avg(CAST({column} AS INTEGER))", "avg"
            continue
        else:
            value = column
            stats = ("min", "max") if column.endswith("T") else ("avg", "max")

        for stat in stats:
            yield f"{prefix}_{column}_{stat}", f"{stat}({value})", stat


def summarize_history_pair(con, source, work_dir):
    parents = read_training_history(con, source, 1)
    details = read_training_history(con, source, 2)
    parent_count = f"{source}_1_record_count"
    detail_count = f"{source}_2_record_count"
    parents_with_details = f"{source}_2_parent_count"

    # Depth 1 tables
    parent_stats = ["case_id", f"count(*) AS {parent_count}"]
    if source == "person":
        parent_stats.extend(
            f"max({column}) FILTER (WHERE num_group1 = 0) AS person_1_applicant_{column}"
            for column in parents.columns if column not in {"case_id", "num_group1"}
        )
    else:
        parent_stats.extend(
            f"{expression} AS {name}" for name, expression, _ in history_statistics(parents, f"{source}_1")
        )
    parent_path = work_dir / f"{source}_1_summary.parquet"
    parents.aggregate(", ".join(parent_stats), "case_id").write_parquet(str(parent_path))

    # Depth 2 tables
    parent_key = "case_id, num_group1"
    detail_stats = [parent_key, "count(*) AS rows_per_parent"]
    case_stats = [
        "case_id",
        f"CAST(sum(rows_per_parent) AS BIGINT) AS {detail_count}",
        f"count(*) AS {parents_with_details}",
        f"max(rows_per_parent) AS {source}_2_max_records_per_parent",
    ]
    for name, expression, rollup in history_statistics(details, f"{source}_2"):
        detail_stats.append(f"{expression} AS {name}")

        if name.endswith("_distinct_count"):
            output_name = f"{name}_max_per_parent"
        elif rollup == "avg":
            output_name = f"{name}_across_parents"
        else:
            output_name = name
        case_stats.append(f"{rollup}({name}) AS {output_name}")

    details_by_parent = details.aggregate(", ".join(detail_stats), parent_key)
    matched_details = details_by_parent.join(parents.project(parent_key), parent_key, how="semi")
    detail_path = work_dir / f"{source}_2_summary.parquet"
    matched_details.aggregate(", ".join(case_stats), "case_id").write_parquet(str(detail_path))
    return (parent_path, detail_path), (parent_count, detail_count, parents_with_details)

In [ ]:
paired_sources = ("applprev", "credit_bureau_a", "credit_bureau_b", "person")
depth2_path = depth1_path.with_name("train_base_depth2.parquet")

# Stage the case-level summaries and allow the large bureau aggregations to spill to disk.
with TemporaryDirectory(dir=depth2_path.parent) as temp_dir:
    work_dir = Path(temp_dir)
    with duckdb.connect(config={
        "threads": 2,
        "preserve_insertion_order": False,
        "memory_limit": "2GB",
        "temp_directory": str(work_dir / "spill"),
    }) as con:
        summaries = []
        count_columns = []
        for source in paired_sources:
            summary_paths, source_counts = summarize_history_pair(con, source, work_dir)
            count_columns.extend(source_counts)
            summaries.extend(summary_paths)
            print(f"Summarized {source}")

        zero_missing_counts = ", ".join(f"coalesce({column}, 0) AS {column}" for column in count_columns)

        # Use the same case partition on both sides of every join.
        batch_count = 8
        batch_paths = []
        for batch in range(batch_count):
            case_filter = f"hash(case_id) % {batch_count} = {batch}"
            train_batch = con.read_parquet(str(depth1_path)).filter(case_filter)
            for path in summaries:
                summary = con.read_parquet(str(path)).filter(case_filter)
                train_batch = train_batch.join(summary, "case_id", how="left")
            train_batch = train_batch.project(f"* REPLACE ({zero_missing_counts})")
            batch_path = work_dir / f"batch_{batch}.parquet"
            train_batch.write_parquet(str(batch_path), row_group_size=32768)
            batch_paths.append(str(batch_path))
            print(f"Joined case batch {batch + 1}/{batch_count}")

        print(f"Combining {len(batch_paths)} case batches into {depth2_path.name}...", flush=True)
        train_depth2 = con.read_parquet(batch_paths)
        staged_output = work_dir / depth2_path.name
        train_depth2.write_parquet(str(staged_output), row_group_size=32768)
        staged_output.replace(depth2_path)

print(f"Saved {depth2_path.resolve()}")